# Sparse Models and Mixture of Experts

Build a token router, measure imbalance and communication, and compare sparse and hybrid architectures for a target workload.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · A Monday morning at HarborLink

HarborLink's engineering team wants to process rider questions and long equipment logs on the same fleet. Arun has a dense baseline, a candidate with sparse experts, and a limited device budget. The support desk only cares which choice still works during a noisy Monday queue.

The workplace scenario is fictional.


## Route tokens through experts


### Lecture 01 · Token routing and top-k experts

Implement top-k expert routing and distinguish it from routing entire requests to different models.

#### Follow one token across the layer

A maintenance request goes to the long-document endpoint. A diagram on the team wall labels that endpoint an expert, which makes it hard to discuss what the model actually computes.

Draw one box around the endpoint decision and another around the token-level operation. A request can take one endpoint and still use many experts inside it across tokens and layers.

#### How it works

A sparse transformer can replace its feed-forward block with a bank of feed-forward experts. A learned router scores experts from each token's hidden state. The layer combines the chosen expert outputs, then returns the result to the transformer. The routing weights can be normalized over the selected experts. A request router operates outside the network and selects a model or workflow for a whole request.

##### What crosses the boundary
Apply the router to a token's hidden state and inspect the scores before interpreting the chosen experts. The scores determine expert identity and can also influence the mixture weights. The residual stream and attention use separate computation. In a model with one shared expert, add that expert's compute for every token before comparing with a dense feed-forward block. Always state which parameters your active-count comparison includes.

A request-level cascade may choose a sparse model as an endpoint. That service decision happens at a different layer from the token router, and you can change it without altering the checkpoint.

#### Worked example

If two experts return 2 and 6 with normalized weights 0.75 and 0.25, the mixed output is 3. With eight equally sized experts and top-2, 25% of the expert-bank parameters are active per token in this toy layer. The attention block and shared parameters still execute.

#### Try it

Change the top-k value and inspect both the output and the two parameter counts.


In [ ]:
from math import exp, isclose


def mix_experts(logits, outputs, k):
    if not 1 <= k <= len(logits) or len(logits) != len(outputs):
        raise ValueError("Choose k from the available experts")
    selected = sorted(range(len(logits)), key=lambda expert: -logits[expert])[:k]
    maximum = logits[selected[0]]
    weights = [exp(logits[expert] - maximum) for expert in selected]
    total = sum(weights)
    return sum(weight * outputs[expert] for expert, weight in zip(selected, weights)) / total


assert isclose(mix_experts((1.0, 0.0), (2.0, 6.0), 1), 2.0)
for k in (1, 2):
    print("output", mix_experts((1.0, 0.0), (2.0, 6.0), k),
          "active", 100 + k * 400, "resident", 100 + 2 * 400)


**Compare your result**

Top-1 selects a single expert's output. Resident expert weights do not shrink when k changes. Add shared weights to both totals when estimating the full model.


### Lecture 02 · Expert capacity and load balancing

Measure routing skew, simulate a capacity limit, and compare overflow against a balancing policy.

#### A burst hits the same expert

A station closure causes many near-identical equipment questions to arrive together. Last week's averaged routing histogram looked balanced.

Inspect assignments at the batch boundary. Count overflow per expert, then rerun the behavior check when assignments are rerouted. A healthier histogram can still hide changed answers.

#### How it works

A batch can favor one expert even when the router looks balanced across a long evaluation. A fixed bucket can overflow. Dropping, delaying, and rerouting assignments have different effects on quality and latency. Training-time balancing losses or bias updates influence routing decisions. Serving-time expert replication and placement change where already-chosen expert work runs.

##### Count assignments per batch
Top-2 over 100 tokens produces 200 assignments if every token reaches two experts. Divide assignment counts by the available capacity and inspect the highest-loaded expert. Weighting the result by the total number of original tokens answers a different question. A capacity limit may be a training design choice, and some serving implementations use dropless kernels. A simulator with overflow demonstrates the consequence of a fixed-capacity policy. It does not imply every implementation drops tokens.

#### Worked example

Six tokens choose expert 0 and two choose expert 1. With a capacity of four per expert, two assignments overflow even though total capacity is eight. Rerouting them fills the second expert but changes which expert processes their tokens.

#### Try it

Compare the balanced and overloaded batches. Try capacity three with the same two experts.


In [ ]:
def dispatch(preferences, expert_count, capacity, reroute=False):
    load = [0] * expert_count
    overflow = changed = 0
    for order in preferences:
        choices = order if reroute else order[:1]
        for expert in choices:
            if load[expert] < capacity:
                load[expert] += 1
                changed += expert != order[0]
                break
        else:
            overflow += 1
    return load, overflow, changed


skewed = [(0, 1)] * 6 + [(1, 0)] * 2
balanced = [(0, 1), (1, 0)] * 4
assert dispatch(skewed, 2, 4) == ([4, 2], 2, 0)
assert dispatch(balanced, 2, 4, reroute=True) == ([4, 4], 0, 0)
assert dispatch(skewed, 2, 4, reroute=True)[1] == 0
for cap in (4, 3):
    print("capacity", cap, dispatch(skewed, 2, cap, reroute=True))


**Compare your result**

At most six assignments fit. Report the remaining overflow and the fallback assignments separately.


## Size the system and workload


### Lecture 03 · Expert parallelism and communication

Account for expert dispatch and return traffic alongside local computation and queueing.

#### Follow the activation across devices

The sparsest candidate reduces expert FLOPs in a spreadsheet. Its busiest expert lives across a slow link from the device handling most inputs.

Estimate the outgoing and return payload. Mark the synchronization cost and collective startup as unknown. Watch the busiest rank before attributing a latency miss to the model math.

#### How it works

Expert parallelism distributes expert weights across devices. A token whose chosen expert is remote must send an activation and receive the transformed activation. Traffic depends on the routing distribution, placement, shared replication, and message layout. A simplified byte count excludes collective startup, padding and synchronization, so it cannot predict latency by itself.

##### Placement belongs in the latency budget
Split weight memory from communication. Replicating a popular expert may reduce load on one rank at the cost of memory elsewhere. Measure collective time, maximum rank load, per-device memory, and throughput with fixed sequence lengths and concurrency. A low average rank load can coexist with a straggler because the step must wait for the slowest participants. Compare a placement change with the same routed workload to isolate its effect.

#### Worked example

For 64 remote assignments of width 4096 at two bytes per number, dispatch and return together move 1 MiB of activation payload under the simple accounting model.

#### Try it

Place the busiest expert on another device and inspect how remote assignment traffic changes. Record one measurement that the byte model lacks.


In [ ]:
def remote_bytes(assignments, placements, width, element_bytes):
    remote = sum(source_rank != placements[expert] for source_rank, expert in assignments)
    return remote * width * element_bytes * 2


traffic = [(0, 0)] * 64 + [(1, 1)] * 8
for placement in ((0, 1), (1, 0)):
    print("payload bytes", remote_bytes(traffic, placement, 4096, 2))
assert remote_bytes([(0, 0)] * 64, (1,), 4096, 2) == 1024 ** 2


**Compare your result**

Ownership and routing together determine traffic. Use a serving profiler to measure dispatch time, device utilization, and the slowest rank before choosing an expert layout.


### Lecture 04 · Hybrid backbones and architecture selection

Compare conventional KV cache growth with a hybrid model that combines recurrent state and full attention.

#### The workload has two kinds of long prompt

Support wants exact citations in a long policy. Maintenance wants a summary of recurring events across a log. Those requests can have the same number of input tokens.

Measure memory against input length and run each task separately. A lower cache estimate can be a useful reason to benchmark and says little about quality on an exact lookup.

#### How it works

MoE changes the feed-forward path. Attention variants change how a layer represents or accesses sequence history. Grouped-query attention shares key and value heads across query heads. Multi-head latent attention stores a compressed latent representation with implementation-specific cache overhead. Hybrid models can mix gated linear or state-space layers with full-attention layers, so the remaining full-attention layers still grow a cache with context.

##### Match architecture to a real request shape
Use one formula per attention or recurrent layer. Grouped-query attention reduces KV heads relative to multi-head attention, while some recurrent layers keep a fixed-size state for each active sequence. Compressed latent KV is a third approach and cache layout varies by runtime. Report actual device memory separately from arithmetic estimates. Measure long-context evidence use as well as memory, and fix the model and tokenizer when isolating a serving-engine change.

#### Worked example

A 32-layer hybrid with eight full-attention layers can grow less token-dependent state than 32 equivalent full-attention layers. The recurrent layers still consume fixed per-sequence state, and the crossover changes with context length.

#### Try it

Compare both formulas at short and long contexts. Add weight memory separately and explain what a real load test must hold constant.


In [ ]:
def estimate_state(tokens, layers, attention_layers, kv_heads=4, head_dim=128, scalar_bytes=2,
                   recurrent_state_bytes=65536):
    kv = tokens * attention_layers * 2 * kv_heads * head_dim * scalar_bytes
    return kv + (layers - attention_layers) * recurrent_state_bytes


for tokens in (128, 4096, 32768):
    print(tokens, {count: estimate_state(tokens, 32, count) for count in (32, 8)})
assert estimate_state(0, 32, 8) == 24 * 65536
assert estimate_state(4096, 32, 8) < estimate_state(4096, 32, 32)


**Compare your result**

Compare quality on exact retrieval and long document tasks alongside throughput and memory. The simulation estimates state bytes under stated assumptions.


## Course project

Submit a sparse-routing simulator and an architecture decision for a fixed interactive and batch workload.

Count active and resident parameters separately. Include a load-skew slice and an overflow policy. Identify which bottleneck needs a real multi-device benchmark.
